In [ ]:
# AI Risk 프레임 분석 - enum 스키마 버전 (최고 성능)
import pandas as pd
import openai
from tqdm import tqdm
import re
import os
import logging
import time
import json

In [ ]:
# 로깅 설정
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.FileHandler("ai_risk_frame_analysis.log"),
              logging.StreamHandler()]
)
logger = logging.getLogger(__name__)

In [ ]:
# AI RISK 프레임 정의 (3개 프레임)
ALLOWED_FRAMES = {
    "대규모 허위정보 감시 및 영향력": {
        "description": "악의적인 목적을 가진 인간이 AI를 활용하여 의도적이고, 조직적으로 여론을 조작하거나 사회적 혼란을 야기할 수 있는 위험. 딥페이크, 선전, 댓글조작 등이 포함"
    },
    "사이버 공격, 무기 개발 또는 사용, 대량 살상": {
        "description": "악의적인 목적을 가진 인간이 AI를 활용하여 물리적 시설이나 인간에게 피해를 가하고자, 공격 수단이나 무기를 개발하는 위험. 사이버 공격, 자율무기 시스템, 생화학무기 설계, 테러 도구 개발 등 실제 파괴나 살상을 목적하는 AI 악용이 포함"
    },
    "사기, 스캠 및 표적 조작": {
        "description": "악의적인 목적을 가진 인간이 사기, 피싱, 신원 도용, 협박 등 특정 개인의 재산이나 사생활을 노리는 범죄 행위에 AI를 활용하는 위험"
    },
}

In [ ]:

def format_allowed_frames():
    """AI Risk 프레임 목록을 프롬프트용 문자열로 변환"""
    frame_list = []
    for i, (frame_name, frame_info) in enumerate(ALLOWED_FRAMES.items(), 1):
        frame_text = f"{i}. {frame_name}: {frame_info['description']}"
        frame_list.append(frame_text)
    return "\n".join(frame_list)

def create_frame_schema():
    """JSON Schema 동적 생성 (enum 제약 포함)"""
    return {
        "type": "object",
        "properties": {
            "frame": {
                "type": "string",
                "description": "AI Risk 프레임 이름 (정확히 하나를 선택)",
                "enum": list(ALLOWED_FRAMES.keys())
            },
            "reason": {
                "type": "string",
                "description": "해당 프레임으로 분류한 근거를 한 문장으로",
                "minLength": 10,
                "maxLength": 200
            }
        },
        "required": ["frame", "reason"],
        "additionalProperties": False
    }

# 단순화된 프롬프트 (JSON Schema가 형식 강제)
PROMPT_TEMPLATE = """다음 텍스트를 분석하여 가장 적절한 AI Risk 프레임을 선택하세요.

[분석할 텍스트]:
{text}

[AI Risk 프레임]:
{allowed_frames}

위 3개 프레임 중 가장 적절한 하나를 선택하고, 그 이유를 한 문장으로 설명하세요."""

def clean_text_for_analysis(text):
    """분석용 텍스트 정리"""
    if pd.isna(text) or text == "":
        return "본문 정보 없음"

    text = str(text)
    text = re.sub(r'\s+', ' ', text).strip()

    if text == "":
        return "본문 정보 없음"

    return text

class NewsFrameAnalyzer:
    def __init__(self, client, output_dir='./', use_schema=True):
        """
        EXAONE 32B GGUF 버전 초기화

        Args:
            client: OpenAI client
            output_dir: 출력 디렉토리
            use_schema: JSON Schema 사용 여부 (True 권장)
        """
        self.client = client
        self.output_dir = output_dir
        self.use_schema = use_schema

        os.makedirs(self.output_dir, exist_ok=True)

        if use_schema:
            logger.info("🚀 EXAONE 32B GGUF 프레임 분석기 초기화 완료! (JSON Schema 사용)")
        else:
            logger.info("🚀 EXAONE 32B GGUF 프레임 분석기 초기화 완료! (JSON Schema 미사용)")

    def analyze_news(self, text):
        """뉴스 텍스트를 분석"""
        allowed_frames_text = format_allowed_frames()
        prompt = PROMPT_TEMPLATE.format(
            text=text[:1000],  # 너무 긴 텍스트 방지
            allowed_frames=allowed_frames_text
        )

        try:
            # API 호출 파라미터 구성
            api_params = {
                "model": "exaone-3.5-32b-instruct",
                "messages": [
                    {
                        "role": "system",
                        "content": "당신은 AI Risk를 정확히 분석하는 전문가입니다."
                    },
                    {"role": "user", "content": prompt}
                ],
                "temperature": 0.0,
                "max_tokens": 300,
            }

            # JSON Schema 사용 시 추가
            if self.use_schema:
                api_params["response_format"] = {
                    "type": "json_schema",
                    "json_schema": {
                        "name": "ai_risk_frame",
                        "strict": True,
                        "schema": create_frame_schema()
                    }
                }

            response = self.client.chat.completions.create(**api_params)

            model_response = response.choices[0].message.content.strip()
            logger.debug(f"EXAONE 32B 응답: {model_response}")

            # JSON 파싱
            result = json.loads(model_response)

            # 필드 검증
            if "frame" not in result or "reason" not in result:
                logger.error(f"필수 필드 누락: {result}")
                return "파싱실패", f"필수 필드 누락"

            frame = result["frame"].strip()
            reason = result["reason"].strip()

            # enum 사용 시 추가 검증 불필요 (자동으로 정확함)
            if self.use_schema:
                # 이미 enum으로 제한되어 있음
                pass
            else:
                # enum 미사용 시 검증 필요
                if frame not in ALLOWED_FRAMES:
                    logger.warning(f"허용되지 않은 프레임: {frame}")
                    return "파싱실패", f"허용되지 않은 프레임: {frame}"

            logger.info(f"✅ 프레임: {frame[:30]}..., 이유: {reason[:50]}...")
            return frame, reason

        except json.JSONDecodeError as e:
            logger.error(f"JSON 파싱 오류: {str(e)}")
            return "파싱실패", f"JSON 파싱 오류: {str(e)}"
        except Exception as e:
            logger.error(f"EXAONE 32B 분석 중 오류: {str(e)}")
            return "파싱실패", f"API 오류: {str(e)}"

    def process_dataframe(self, df, output_file='frame_analysis_results.csv'):
        """데이터프레임 처리"""
        output_path = os.path.join(self.output_dir, output_file)
        checkpoint_path = os.path.join(self.output_dir, f"checkpoint_{output_file}")

        frame_names = []
        frame_reasons = []

        # 체크포인트 복구
        start_idx = 0
        if os.path.exists(checkpoint_path):
            try:
                checkpoint_df = pd.read_csv(checkpoint_path, encoding='utf-8-sig')
                if '프레임 이름' in checkpoint_df.columns and '이유' in checkpoint_df.columns:
                    completed_rows = sum(~checkpoint_df['프레임 이름'].isna())
                    if completed_rows > 0:
                        start_idx = completed_rows
                        logger.info(f"📂 체크포인트에서 복구: {start_idx}행부터 계속")
                        frame_names = checkpoint_df['프레임 이름'].tolist()[:start_idx]
                        frame_reasons = checkpoint_df['이유'].tolist()[:start_idx]
            except Exception as e:
                logger.warning(f"체크포인트 복구 실패: {str(e)}")

        # 진행 상황 표시
        pbar = tqdm(total=len(df) - start_idx, initial=0, desc="🚀 EXAONE 32B 프레임 분석")

        # 데이터 순차 처리
        for i in range(start_idx, len(df)):
            text = clean_text_for_analysis(df.iloc[i]['chunk'])

            try:
                # 최대 3번 재시도
                frame, reason = "파싱실패", "처리 실패"
                for attempt in range(3):
                    try:
                        frame, reason = self.analyze_news(text)

                        # 파싱 성공 확인
                        if frame != "파싱실패":
                            break

                        if attempt < 2:
                            logger.warning(f"행 {i} 재시도 {attempt+1}/3")
                            time.sleep(1)
                    except Exception as e:
                        if attempt < 2:
                            logger.warning(f"행 {i} 처리 실패 (시도 {attempt+1}/3): {str(e)}")
                            time.sleep(1)
                        else:
                            logger.error(f"행 {i} 최대 재시도 초과: {str(e)}")
                            frame, reason = "파싱실패", f"처리 실패: {str(e)}"
            except Exception as e:
                logger.error(f"행 {i} 처리 중 예외 발생: {str(e)}")
                frame, reason = "파싱실패", f"처리 오류: {str(e)}"

            frame_names.append(frame)
            frame_reasons.append(reason)
            pbar.update(1)

            # 체크포인트 저장 (10개 항목마다)
            if (i + 1) % 10 == 0 or i == len(df) - 1:
                temp_df = df.copy()
                temp_df['프레임 이름'] = frame_names + [''] * (len(temp_df) - len(frame_names))
                temp_df['이유'] = frame_reasons + [''] * (len(temp_df) - len(frame_reasons))
                temp_df.to_csv(checkpoint_path, encoding='utf-8-sig', index=False)
                logger.info(f"💾 체크포인트 저장: {len(frame_names)}/{len(df)}")

        pbar.close()

        # 최종 결과 저장
        result_df = df.copy()
        result_df['프레임 이름'] = frame_names
        result_df['이유'] = frame_reasons

        # 프레임 분포 출력
        frame_distribution = result_df['프레임 이름'].value_counts()
        logger.info(f"\n📊 EXAONE 32B 프레임 분포:\n{frame_distribution}")

        result_df.to_csv(output_path, encoding='utf-8-sig', index=False)
        logger.info(f"🎉 분석 완료! 결과가 {output_path}에 저장되었습니다.")

        return result_df

In [ ]:
def test_single_news(text, client, use_schema=True):
    """단일 뉴스 테스트용 함수"""
    analyzer = NewsFrameAnalyzer(client, output_dir='/mnt/c/Users/com/Downloads/', use_schema=use_schema)
    frame, reason = analyzer.analyze_news(text)

    print(f"\n{'='*60}")
    print(f"📰 뉴스: {text[:100]}...")
    print(f"🎯 프레임: {frame}")
    print(f"💡 이유: {reason}")
    print(f"{'='*60}\n")

    return frame, reason

In [ ]:
# 메인 실행 코드
if __name__ == "__main__":
    # EXAONE 32B GGUF 연결 설정
    print("🔷 EXAONE 32B GGUF 연결 중...")

    client = openai.OpenAI(
        base_url="http://218.156.133.228:1234/v1",
        api_key="not-needed"
    )

    # 연결 테스트
    try:
        test_response = client.chat.completions.create(
            model="exaone-3.5-32b-instruct",
            messages=[{"role": "user", "content": "AI Risk란 무엇인가요? 128토큰 이내로 설명해주세요."}],
            max_tokens=128
        )
        print("✅ EXAONE 32B 연결 성공!")
    except Exception as e:
        print(f"❌ 연결 실패: {e}")
        exit(1)

    # 데이터 로드
    print("\n📊 데이터 로드 중...")
    file_path = "/mnt/c/Users/com/Downloads/악의적 행위자의 오용(malicious actors & misuse)2.1.csv"
    df = pd.read_csv(file_path, encoding="utf-8-sig")

    print(f"✅ 데이터 로드 완료: {len(df)}개의 기사")
    print(f"   - chunk 열 NaN: {df['chunk'].isna().sum()}개")
    print(f"   - chunk 열 빈 문자열: {(df['chunk'] == '').sum()}개")

    # 텍스트 정제
    df['chunk'] = df['chunk'].apply(clean_text_for_analysis)

    # 테스트 샘플 실행
    test_text = """
    챗GPT가 보여준 생성 AI의 충격만큼이나, 드리울 그늘도 짙다. 저작권 분쟁, 해킹 악용, 혐오 표현 재생산 등 풀어야 할 숙제가 많아서다.
    """

    print("\n" + "="*60)
    print("🧪 테스트 샘플 실행 (JSON Schema 사용)")
    print("="*60)
    test_single_news(test_text, client, use_schema=True)

    # 전체 분석 실행
    print("\n" + "="*60)
    print("🎯 전체 데이터 분석 시작 (JSON Schema 사용)")
    print("="*60 + "\n")

    # use_schema=True로 설정 (권장)
    analyzer = NewsFrameAnalyzer(client, output_dir='/mnt/c/Users/com/Downloads/', use_schema=True)

    result_df = analyzer.process_dataframe(
        df,
        output_file='악의적 행위자의 오용(malicious actors & misuse)2.1+결과_enum스키마.csv'
    )

    # 결과 요약
    print("\n" + "="*60)
    print("📊 최종 결과 요약")
    print("="*60)

    frame_counts = result_df['프레임 이름'].value_counts()
    print(frame_counts)

    # 분류 성공률 확인
    total_count = len(result_df)
    parse_failed_count = frame_counts.get('파싱실패', 0)
    success_rate = ((total_count - parse_failed_count) / total_count) * 100

    print(f"\n🎯 분류 성공률: {success_rate:.2f}% ({total_count - parse_failed_count}/{total_count})")
    print(f"❌ 파싱 실패: {parse_failed_count}개")

    print("\n" + "="*60)
    print("✅ 분석 완료!")
    print("="*60)